# 경기도 카드 소비 데이터 매출 예측 모델 학습 (전체 데이터)

- 수치형: age, day, hour, month, cnt → Label Encoding
- 범주형: sex, admi_cty_no, card_tpbuz_nm_1, card_tpbuz_nm_2 → One-Hot Encoding (sparse)
- sparse OHE로 메모리 절약 (51M × 494컬럼 dense ≈ 100GB → sparse ≈ 1GB)

In [ ]:
import os, glob, time, gc
import numpy as np
import pandas as pd
import joblib
import scipy.sparse as sp
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

DATA_DIR   = "/kaggle/input/gyeonggi-card-data"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(f"{OUTPUT_DIR}/model",    exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/encoders", exist_ok=True)

files = sorted(glob.glob(f"{DATA_DIR}/tbsh_gyeonggi_day_*.csv"))
print(f"파일 수: {len(files)}개")

In [ ]:
# =========================================================
# 설정
# =========================================================
NUM_COLS  = ["age", "day", "hour", "month", "cnt"]   # Label Encoding
CAT_COLS  = ["sex", "admi_cty_no", "card_tpbuz_nm_1", "card_tpbuz_nm_2"]  # OHE
ALL_FEATS = NUM_COLS + CAT_COLS
REMOVE_OUTLIER = True
SAMPLE_PER_FILE = 70000  # 147파일 × 70K = ~10M행 → sparse OHE 메모리 안전

DTYPES = {
    "admi_cty_no":     "int32",
    "hour":            "int8",
    "sex":             "category",
    "age":             "int8",
    "day":             "int8",
    "amt":             "int64",
    "cnt":             "int32",
    "card_tpbuz_nm_1": "category",
    "card_tpbuz_nm_2": "category",
}

In [ ]:
# =========================================================
# 1) 전체 데이터 로드 (파일당 샘플링)
# =========================================================
dfs = []
t0 = time.time()
for i, f in enumerate(files, 1):
    city = os.path.basename(f).split("_")[-1].replace(".csv", "")
    csv_cols = [c for c in ALL_FEATS if c != "month"]  # month는 ta_ymd에서 파생
    df = pd.read_csv(f, encoding="utf-8-sig",
                     usecols=["ta_ymd"] + csv_cols + ["amt"],
                     dtype=DTYPES)
    df["month"] = df["ta_ymd"].astype(str).str[4:6].astype("int8")
    df["log_amt"] = np.log1p(df["amt"]).astype("float32")
    if REMOVE_OUTLIER:
        upper = df["amt"].quantile(0.99)
        df = df[df["amt"] <= upper]
    df = df[ALL_FEATS + ["log_amt"]].dropna()
    if len(df) > SAMPLE_PER_FILE:
        df = df.sample(SAMPLE_PER_FILE, random_state=42)
    dfs.append(df)
    if i % 10 == 0 or i == len(files):
        print(f"[{i}/{len(files)}] {city}  ({time.time()-t0:.0f}s)  {len(df):,}행")

df_all = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()
for col in CAT_COLS:
    df_all[col] = df_all[col].astype("category")
print(f"전체: {len(df_all):,}행  /  메모리: {df_all.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
# =========================================================
# 2) 인코딩
#    - NUM_COLS: Label Encoding (순서 의미 있음)
#    - CAT_COLS: One-Hot Encoding sparse (순서 의미 없음)
# =========================================================
label_encoders = {}
for col in ["age", "day", "hour", "month"]:  # cnt는 수치형 그대로
    le = LabelEncoder()
    df_all[col] = le.fit_transform(df_all[col].astype(str))
    label_encoders[col] = le

# sparse OHE — 메모리 핵심
ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
ohe_sparse = ohe.fit_transform(df_all[CAT_COLS].astype(str))

# 수치형 컬럼을 sparse로 변환 후 합치기
num_sparse = sp.csr_matrix(df_all[NUM_COLS].values.astype("float32"))
X = sp.hstack([num_sparse, ohe_sparse], format="csr")

ohe_names = ohe.get_feature_names_out(CAT_COLS).tolist()
feature_columns = NUM_COLS + ohe_names

y = df_all["log_amt"].values
del df_all, num_sparse, ohe_sparse; gc.collect()

print(f"피처 수: {len(feature_columns)}개")
print(f"X shape: {X.shape}  /  sparse 밀도: {X.nnz / (X.shape[0]*X.shape[1]):.4f}")

In [ ]:
# =========================================================
# 3) 학습 / 검증 분할
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
del X, y; gc.collect()
print(f"학습: {X_train.shape[0]:,}행 / 검증: {X_test.shape[0]:,}행")

In [ ]:
# =========================================================
# 4) LightGBM 학습
# =========================================================
t0 = time.time()
model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100)
    ]
)
print(f"학습 완료: {time.time()-t0:.1f}초")

In [ ]:
# =========================================================
# 5) 평가
# =========================================================
pred = model.predict(X_test)
y_real    = np.expm1(y_test)
pred_real = np.maximum(np.expm1(pred), 0)

r2   = r2_score(y_real, pred_real)
rmse = np.sqrt(mean_squared_error(y_real, pred_real))
mae  = mean_absolute_error(y_real, pred_real)
print(f"R²   = {r2:.4f}")
print(f"RMSE = {rmse:,.0f}")
print(f"MAE  = {mae:,.0f}")

In [ ]:
# =========================================================
# 6) 저장
# =========================================================
joblib.dump(model,          f"{OUTPUT_DIR}/model/sales_predict_model.pkl")
joblib.dump(label_encoders, f"{OUTPUT_DIR}/encoders/label_encoders.pkl")
joblib.dump(ohe,            f"{OUTPUT_DIR}/encoders/onehot_encoder.pkl")
joblib.dump(feature_columns,f"{OUTPUT_DIR}/encoders/feature_columns.pkl")
joblib.dump({
    "model_name":      "LightGBM",
    "R2": r2, "RMSE": rmse, "MAE": mae,
    "total_rows":      X_train.shape[0] + X_test.shape[0],
    "features":        feature_columns,
    "num_cols":        NUM_COLS,
    "cat_cols":        CAT_COLS,
    "use_log_target":  True,
    "remove_outliers": REMOVE_OUTLIER,
    "encoding":        "ohe",
}, f"{OUTPUT_DIR}/model/model_info.pkl")

print("저장 완료! Output 탭에서 다운로드:")
print("  model/sales_predict_model.pkl, model/model_info.pkl")
print("  encoders/label_encoders.pkl, onehot_encoder.pkl, feature_columns.pkl")